# Self-supervised pretraining for PVT v2: SimMIM (default) or JEPA

`train.py --task ssl` runs the same thing from a terminal; this notebook is the
interactive front end. Read **`docs/SIMMIM_GUIDE.md`** first (recipe, the
overlapping-patch-embed leak and the `mask_space` choice, the three-path
pretraining ablation, the evaluation protocol); `docs/JEPA_GUIDE.md` covers the
alternative method (`ssl.method: "jepa"`).

- **SimMIM** (Xie et al., CVPR 2022; microsoft/SimMIM @ d3e29bc): random 32-px
  patches, 60% masked, a learnable mask token after the stage-1 embed, a 1x1 conv +
  PixelShuffle(32) head on the 7x7 stage-4 map, L1 on the masked pixels only.
  AdamW 2e-4 x batch/512, warmup 10 ep, cosine to 1e-5 x batch/512, wd 0.05,
  betas (0.9, 0.999), clip 5, drop path 0, crop + flip only.
- **Three pretraining paths** (`docs/SIMMIM_GUIDE.md` §6): dense pretrain -> dense
  fine-tune; MoE pretrain (`use_moe: True`, router trained under the SSL loss) ->
  MoE fine-tune (straight load); dense pretrain -> MoE fine-tune (experts upcycled
  at fine-tune time). Dense is the default below.
- **Chain**: pretrain -> `train.py --recipe ssl_finetune` (supervised ImageNet-1k,
  the intermediate stage, SwinV2 §4.2 / BEiT) -> `--recipe downstream`. Every
  `results.json` records the chain that produced its numbers.
- **Evaluation**: `results.json` in the run directory every epoch; k-NN and the
  linear probe via `evaluate.py` are collapse detectors — under masked image
  modelling they are EXPECTED to read low; the headline is the fine-tuned top-1.

No MoE backend is needed for the dense paths; MoE pretraining needs Tutel (or
`--backend native`).


In [ ]:
# Environment check — plain Jupyter on B200 (sm_100) or RTX 5090 (sm_120).
# Credentials: export HF_TOKEN / WANDB_API_KEY in the shell that starts Jupyter.
import importlib
import subprocess
import sys

import torch

print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})")
else:
    print("WARNING: no GPU visible — training will not be practical.")

_torch_minor = tuple(int(p) for p in torch.__version__.split("+")[0].split(".")[:2])
if _torch_minor < (2, 5):
    print("NOTE: torch >= 2.5 recommended (fast SDPA GQA path; >=2.4 for fused RMSNorm). "
          "The code falls back gracefully but slower.")

_required = ["pytorch_lightning", "torchmetrics", "timm", "datasets", "transformers",
             "huggingface_hub", "pandas", "matplotlib", "fvcore", "wandb"]
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
if _missing:
    print(f"Installing: {_missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

In [ ]:
# Make the repo importable (notebooks/ lives one level under the repo root).
import pathlib
import sys

cwd = pathlib.Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
assert (REPO_ROOT / "pvt_moe").is_dir(), f"pvt_moe package not found under {REPO_ROOT}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pvt_moe import default_config, merge_config, validate_config
from pvt_moe.data import build_dataloaders
from pvt_moe.engine import build_ssl_trainer, setup_environment
from pvt_moe.ssl import (
    SimMIMMaskGenerator,
    backbone_filename,
    build_ssl_module,
    mask_token_routing,
    sample_batch_masks,
)

print(f"pvt_moe loaded from {REPO_ROOT}")


In [ ]:
# ============================== CONFIG =====================================
cfg = merge_config(default_config(), {
    "task": "ssl",                   # unlocks unlabelled corpora (PASS)
    "mode": "scratch",
    "use_wandb": True,
    "wandb_project": "pvt-ssl-pretrain",
    "batch_size": 128,               # micro-batch; effective_batch_size (1024) sets the accumulation
    "num_workers": 12,
    # run_name is DERIVED (sv1_b1_pass_r224_dense_rope-s4b1_ln_simmim200); set it
    # only to override.

    "model": {
        "pretrained_hf_id": None,     # SSL trains from scratch
        "ablation": {
            # Dense pretraining is the default (paths 1 and 3 of the three-path
            # ablation). Path 2 = True: the routed layer and its router train
            # under the SimMIM loss, and the fine-tune loads them as trained.
            "use_moe": False,
            "moe_placement": [[], [], [], [-1]],
            "use_rope": True,         # positional signal for stage 4
            # ALL blocks of the last stage, whatever the variant's depth
            # (a literal [0, 1] would be "the first two" of B2's three).
            "rope_last_n_stages": 1,
            "rope_theta": None,   # None = per-mode default (mixed 10 / axial 50)
        },
    },
    # PASS (yukimasano/pass): 1.44M unlabelled images, CC-BY 4.0, no people,
    # from YFCC-100M — the SSL arm's point is CLEAN PROVENANCE, hence the
    # default. Switch to "imagenet-1k" to ask a different question: whether the
    # MoE gain survives SSL pretraining on the same images the supervised arms
    # see. PASS has a single 'train' split: no validation loader; labels enter
    # the pipeline only at evaluate.py / the intermediate fine-tune.
    "dataset": {"name": "pass"},

    "ssl": {
        "method": "simmim",          # "simmim" (default) | "jepa"
        "epochs": 200,               # SimMIM: 100 for a quick arm, 800 for the paper's full run
        "mask_space": "token",       # SimMIM's; "pixel" removes the overlapping-embed leak (guide §3)
        # Everything else — base_lr 2e-4 @ 512 (scaled to the effective batch),
        # warmup 10 ep from 1e-6, final 1e-5, wd 0.05, betas (0.9, 0.999),
        # clip 5, mask 32 px / ratio 0.6 — comes from config.SSL_METHOD_DEFAULTS
        # and is printed by the module at construction. Set a key here only to
        # override it deliberately.
    },
})

cfg = validate_config(cfg)
print("run:", cfg["run_name"], "| chain:", cfg["chain"])


In [ ]:
device = setup_environment(cfg)

In [ ]:
# SSL data: crop + flip only (build_dataloaders swaps the train transform; the
# crop scale follows the method: SimMIM (0.67, 1), JEPA (0.3, 1)).
# val_loader is None for PASS (train split only); pretraining never validates.
train_loader, val_loader = build_dataloaders(cfg, ssl=True)
xb, _ = next(iter(train_loader))
print("batch:", xb.shape)


In [ ]:
# Visualize the masking on the 7x7 patch grid (32 px per cell at 224).
import matplotlib.pyplot as plt
import torch

if cfg["ssl"]["method"] == "simmim":
    gen = SimMIMMaskGenerator(cfg["dataset"]["img_size"], cfg["ssl"]["mask_patch_size"], 4, cfg["ssl"]["mask_ratio"])
    masks, _ = gen(8, generator=torch.Generator().manual_seed(0))
    title = f"SimMIM: {gen.mask_count}/{gen.token_count} patches of {gen.mask_patch_size}px per image"
else:
    masks = sample_batch_masks(8, grid=7, generator=torch.Generator().manual_seed(0)).view(8, 7, 7)
    title = "JEPA multi-block masks"
fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
for ax, m in zip(axes, masks):
    ax.imshow(m, cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"{m.float().mean():.0%}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"{title} (dark = masked / predicted)")
plt.show()


In [ ]:
# Sanity: one masked forward + loss on a small batch before committing GPU-days.
# The module prints the resolved LR rule (base_lr x effective / reference) first.
module = build_ssl_module(cfg).to(device)
print(module.sanity_step(xb[:4].to(device)))
if cfg["model"]["ablation"]["use_moe"]:
    # Path 2: how the router treats masked-position vs visible tokens, and the
    # reminder that the load-balancing loss counts both populations.
    mask_token_routing(module, xb[:16].to(device))


In [ ]:
trainer = build_ssl_trainer(cfg)
trainer.fit(module, train_loader)
# results.json / results.md, last.ckpt, rope_freqs_*.pt land in the run directory every epoch.


In [ ]:
import os

backbone_path = os.path.join(cfg["checkpoint_root"], cfg["run_name"], backbone_filename(cfg["ssl"]["method"]))
module.save_backbone(backbone_path)   # encoder + config (chain provenance) for --recipe ssl_finetune


In [ ]:
# Collapse check on a LABELLED set: k-NN + linear probe on ImageNet-1k.
# Under masked image modelling both are EXPECTED to be low (SimMIM/MAE/BEiT
# all report weak probes and strong fine-tuning) — a probe near chance means
# the encoder learned nothing, a probe well above it means it did; the
# headline number is the fine-tuned top-1. The same via the terminal:
#   python evaluate.py --ckpt <backbone_path> --dataset imagenet-1k --data-dir <arrow> --knn --probe-epochs 20
from pvt_moe.eval.runner import evaluate

eval_result = evaluate(backbone_path, dataset="imagenet-1k", knn=True, probe_epochs=20,
                       num_workers=cfg["num_workers"])


## Handoff: the intermediate supervised stage, then downstream

Pyramid ViTs under masked image modelling go **SSL -> supervised ImageNet-1k ->
downstream** (SwinV2 §4.2, BEiT); `docs/SIMMIM_GUIDE.md` §5 has the citations
and §6 the three pretraining paths. From the terminal:

```bash
# intermediate stage: supervised ImageNet-1k fine-tune of the encoder
#   (SimMIM's 100-ep fine-tune recipe: base_lr 1.25e-3 @ 512, warmup 20,
#    layer_decay 0.9, drop path 0.1; MoE upcycled from the encoder's own FFN
#    unless the encoder already carries experts)
python train.py --recipe ssl_finetune --ckpt <checkpoint_root>/<run_name>/simmim_backbone.pt \
    --dataset imagenet-1k --data-dir /data/imagenet_arrow

# downstream: the small set, fixed epoch budget from the registry
python train.py --recipe downstream --dataset eurosat --data-dir /data/eurosat_arrow \
    --ckpt <checkpoint_root>/<fine-tune run>/last.ckpt
```

The fine-tune must use the SAME variant, RoPE mode and placement as the
pretraining config (here RoPE in every stage-4 block: `--rope-last-n 1`);
`ssl_init` refuses a mismatch (`model.ssl_init_check_arch: false` downgrades
that to a warning). Its run name carries the parent (`..._sslft100_from-dense-simmim200`),
so path 2 and path 3 fine-tunes never share a directory, and every `results.json`
records the full chain. Compare arms with `python tools/compare_runs.py <checkpoint_root>`.
